In [ ]:
# GEO query and DEG analysis script

# Install dependencies if not already installed:
!pip install GEOparse pandas matplotlib seaborn scipy numpy

import GEOparse
import pandas as pd
import numpy as np
from scipy.stats import ttest_ind
import matplotlib.pyplot as plt
import seaborn as sns
import GEOparse
import pandas as pd
gse_id = "GSE47962"  # <-- Change GEO Series
gse = GEOparse.get_GEO(geo=gse_id, destdir=".")

print(f"Dataset: {gse_id}")
print(f"Title: {gse.metadata['title'][0]}")
print(f"Samples: {len(gse.gsms)}")
dfs = []
for gsm_name, gsm in gse.gsms.items():
    if gsm.table is not None and "VALUE" in gsm.table.columns:
        df = gsm.table[["ID_REF", "VALUE"]].copy()
        df.rename(columns={"VALUE": gsm_name}, inplace=True)
        dfs.append(df)

expr_df = dfs[0]
for df in dfs[1:]:
    expr_df = expr_df.merge(df, on="ID_REF")

expr_df = expr_df.drop_duplicates(subset="ID_REF")
expr_df.set_index("ID_REF", inplace=True)

print("Expression Matrix Shape:", expr_df.shape)
expr_df.to_csv("expression.csv")
import numpy as np
import pandas as pd
from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests

n = len(expr_df.columns)
control = expr_df.iloc[:, :n//2]
treated = expr_df.iloc[:, n//2:]

pvals, logFCs, t_stats, avg_exp_values = [], [], [], []

for gene in expr_df.index:
    control_values = control.loc[gene]
    treated_values = treated.loc[gene]

    if control_values.dropna().empty or treated_values.dropna().empty:
        stat, p, logFC, avg_exp = np.nan, np.nan, np.nan, np.nan
    else:
        stat, p = ttest_ind(control_values, treated_values, nan_policy="omit")
        logFC = treated_values.mean() - control_values.mean()
        avg_exp = expr_df.loc[gene].mean()

    t_stats.append(stat)
    pvals.append(p)
    logFCs.append(logFC)
    avg_exp_values.append(avg_exp)

deg_df = pd.DataFrame({
    "Gene": expr_df.index,
    "logFC": logFCs,
    "t": t_stats,
    "Pvalue": pvals,
    "AvgExp": avg_exp_values
})

# Safe log10
deg_df["-log10p"] = -np.log10(deg_df["Pvalue"].replace(0, np.nan))

# Adjust p-values
valid = deg_df["Pvalue"].notna()
if valid.sum() > 1:
    deg_df.loc[valid, "adjPvalue"] = multipletests(deg_df.loc[valid, "Pvalue"], method='fdr_bh')[1]
else:
    deg_df["adjPvalue"] = np.nan

deg_df.to_csv("DEG.csv", index=False)
print("✅ Saved: DEG.csv")

20-Apr-2026 05:44:12 DEBUG utils - Directory . already exists. Skipping.
DEBUG:GEOparse:Directory . already exists. Skipping.
20-Apr-2026 05:44:12 INFO GEOparse - Downloading ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE47nnn/GSE47962/soft/GSE47962_family.soft.gz to ./GSE47962_family.soft.gz
INFO:GEOparse:Downloading ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE47nnn/GSE47962/soft/GSE47962_family.soft.gz to ./GSE47962_family.soft.gz
100%|██████████| 44.0M/44.0M [00:01<00:00, 32.5MB/s]
20-Apr-2026 05:44:14 DEBUG downloader - Size validation passed
DEBUG:GEOparse:Size validation passed
20-Apr-2026 05:44:14 DEBUG downloader - Moving /tmp/tmp_5zvtbdh to /content/GSE47962_family.soft.gz
DEBUG:GEOparse:Moving /tmp/tmp_5zvtbdh to /content/GSE47962_family.soft.gz
20-Apr-2026 05:44:14 DEBUG downloader - Successfully downloaded ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE47nnn/GSE47962/soft/GSE47962_family.soft.gz
DEBUG:GEOparse:Successfully downloaded ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE47nnn/GSE4796

Dataset: GSE47962
Title: SHAE004: SARS-CoV, SARS-dORF6 and SARS-BatSRBD infection of HAE cultures.
Samples: 134
Expression Matrix Shape: (32388, 134)
✅ Saved: DEG.csv
